# 3.1 Creating Synthetic Data

In this example, we show how to create a synthetic data set.

In [ ]:
#__ Import: Numerical __
import numpy as np

#__ Import: Topophone2 __
from topophone2.modelling import ArrayCovarianceModel
from topophone2.utils.read import convert_to_txt

Hello from topophone2!


### 3.1.1 Receiver geometry and frequency vector

Synthetic data requires a 2D array of receiver geometry and a 1D array of frequencies.

In [10]:
#__ Create receiver geometry __
x = np.array([          # Receiver geometry [m] (receiver, direction [XYZ])
    [0,    0,     0],   #     Receiver #1
    [0,    300,   12],  #     Receiver #2
    [260,  150,   8],   #     ...
    [-260, -150,  25],
    [866,  500,   60],
    [0,    -1000, -30],
    [-866, 500,   45],
    [1500, 0,     90]
])

#__ Create frequency vector __
frequency = np.linspace(0.5, 4.0, 15)  # Frequency vector [Hz] (frequency,)

### 3.1.2 Signal content

The signal content is specified by a list, with available components given in the documentation for ``modelling.ArrayCovarianceModel``.

In [ ]:
#__ Signal components __
sigs = [
    'point',               # Plane wave
    'normal',              # Normally distributed wavefield
    dict(
        sig = 'isonoise',  # Isotropic noise field
        dim = 2            # ... in horizontal plane
    )
]

The true model parameters for the synthetic test are given as a 2D vector ``m`` with length 1 along the first axis. The values correspond to the parameter set for the example above:
| Signal component | Parameter set |
|---|---|
|``'point'`` | $\left(E, \xi_1, \xi_2, \xi_3\right)$ |
|``'normal'`` | $\left(E, \xi_1, \xi_2, \xi_3, \sigma_1, \sigma_2, \sigma_3, \rho_{12}, \rho_{13}, \rho_{23}\right)$ |
|``'isonoise'`` | $\left(E, s\right)$ |

In [ ]:
#__ True parameters __
m = np.array([[
    1.0, -2.5e-3, -1.4e-3, -0.5e-3,                                          # 'point'     [energy, xi_x, xi_y, xi_z]
    2.0, 0.9e-3,  2.5e-3,  -1.2e-3, 0.3e-3, 0.2e-3, 0.2e-3, 0.4, 0.0, -0.2,  # 'normal'    [energy, xi_x, xi_y, xi_z, sigma_x, sigma_y, sigma_z, rho_xy, rho_xz, rho_yz]
    0.3, 2.9e-3                                                              # 'isonoise'  [energy, s]
]])

### 3.1.3 Synthesise signal

To create the synthetic signal, first instantiate the forward modeller class ``modelling.ArrayCovarianceModel``, and run the forward model using the method ``forward``. The axes of the sample-covariance matrix are flattened to its upper triangle to save space, so it must be expanded using the method ``expand``. 

In [15]:
#__ Forward modeller __
array_covariance_model = ArrayCovarianceModel(
    x    = x,
    freq = 2 * np.pi * frequency,
    sigs = sigs,
    calc_noise_only = True
)

#__ Run forward model (model, frequency, receiver pair) __
Gamma_flat, _, Gamma_noise_flat, _ = array_covariance_model.forward(m)

#__ Expand axes (model, frequency, receiver, receiver) __
Gamma, _       = array_covariance_model.expand(Gamma_flat, None)
Gamma_noise, _ = array_covariance_model.expand(Gamma_noise_flat, None)

#__ Take away model axis (size 1) __
Gamma, Gamma_noise = Gamma[0], Gamma_noise[0]

#__ Pretend number of observations __
N       = 16
N_noise = 32

### 3.1.4 Save to Topophone2 input file

The synthetic data can be saved in the _Topophone2_ file format using ``utils.read.convert_to_txt``.

In [ ]:
#__ Write to file __
convert_to_txt(
    frequency   = frequency,
    x           = x,
    Gamma_obs   = Gamma,
    output_path = '../data/eg3/eg3_full',
    N_obs       = N
)

Data info:
>>> Number of frequency points (N_freq): 15
>>> Number of receiver positions (N_rec): 8
>>> Number of coordinate dimensions (N_dim): 3
>>> Number of observations (N_obs): 16

Flattening Gamma_obs in 'C' order (receiver q varies fastest, then receiver p, then frequency).

Data successfully written to: ../data/eg3_full_topophone2.txt


In [ ]:
#__ Write to file (noise-only) __
convert_to_txt(
    frequency   = frequency,
    x           = x,
    Gamma_obs   = Gamma_noise,
    output_path = '../data/eg3/eg3_noise',
    N_obs       = N_noise
)

Data info:
>>> Number of frequency points (N_freq): 15
>>> Number of receiver positions (N_rec): 8
>>> Number of coordinate dimensions (N_dim): 3
>>> Number of observations (N_obs): 32

Flattening Gamma_obs in 'C' order (receiver q varies fastest, then receiver p, then frequency).

Data successfully written to: ../data/eg3_noise_topophone2.txt
